In [1]:
!git clone https://github.com/abachaa/MedQuAD.git

Cloning into 'MedQuAD'...
remote: Enumerating objects: 11310, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 11310 (delta 7), reused 5 (delta 5), pack-reused 11300 (from 1)
Receiving objects: 100% (11310/11310), 11.01 MiB | 14.70 MiB/s, done.
Resolving deltas: 100% (6807/6807), done.
Updating files: 100% (11277/11277), done.


In [2]:
# List the contents of the downloaded directory
import os

medquad_path = './MedQuAD'
print("Contents of MedQuAD directory:")
print(os.listdir(medquad_path))

# Explore the 'dataset' subdirectory
dataset_path = os.path.join(medquad_path, 'dataset')
if os.path.exists(dataset_path):
    print("\nContents of MedQuAD/dataset directory:")
    print(os.listdir(dataset_path))
else:
    print("\n'dataset' directory not found inside MedQuAD.")

Contents of MedQuAD directory:
['LICENSE.txt', '8_NHLBI_QA_XML', '10_MPlus_ADAM_QA', '6_NINDS_QA', '2_GARD_QA', '9_CDC_QA', '11_MPlusDrugs_QA', '7_SeniorHealth_QA', '4_MPlus_Health_Topics_QA', '12_MPlusHerbsSupplements_QA', '3_GHR_QA', '1_CancerGov_QA', 'QA-TestSet-LiveQA-Med-Qrels-2479-Answers.zip', '.git', 'readme.txt', '5_NIDDK_QA']

'dataset' directory not found inside MedQuAD.


### Exploring a specific QA directory

Since a top-level 'dataset' directory wasn't found, we'll look inside one of the source-specific directories to understand the data format. Let's start with `1_CancerGov_QA`.

In [3]:
import os

cancer_gov_path = os.path.join(medquad_path, '1_CancerGov_QA')

if os.path.exists(cancer_gov_path):
    print(f"Contents of {cancer_gov_path} directory:")
    print(os.listdir(cancer_gov_path))

    # Optionally, read the first file to get a glimpse of the content
    # Assuming files are XML based on directory names like '8_NHLBI_QA_XML'
    if os.listdir(cancer_gov_path):
        first_file = os.path.join(cancer_gov_path, os.listdir(cancer_gov_path)[0])
        print(f"\nContent of the first file ({os.path.basename(first_file)}):\n")
        with open(first_file, 'r', encoding='utf-8') as f:
            # Read only the first few lines to avoid printing a very large file
            for i, line in enumerate(f):
                print(line.strip())
                if i >= 10:  # Print first 10 lines
                    break
    else:
        print(f"The directory {cancer_gov_path} is empty.")
else:
    print(f"The directory {cancer_gov_path} does not exist.")

Contents of ./MedQuAD/1_CancerGov_QA directory:
['0000013_3_3.xml', '0000019_3.xml', '0000013_3_2.xml', '0000013_2_2.xml', '0000022_1.xml', '0000009_2.xml', '0000005_2.xml', '0000030_1.xml', '0000004_6.xml', '0000004_5.xml', '0000019_4.xml', '0000024_3.xml', '0000003_3.xml', '0000021_2.xml', '0000004_7.xml', '0000001_7.xml', '0000013_2_1.xml', '0000027_4.xml', '0000036_1.xml', '0000040_1.xml', '0000014_3.xml', '0000041_1.xml', '0000034_1.xml', '0000004_1.xml', '0000006_2.xml', '0000015_1.xml', '0000006_8.xml', '0000007_5.xml', '0000013_2_5.xml', '0000039_1.xml', '0000019_5.xml', '0000037_1.xml', '0000014_2.xml', '0000024_10.xml', '0000001_5.xml', '0000027_5.xml', '0000036_3.xml', '0000013_3_4.xml', '0000028_1.xml', '0000001_2.xml', '0000006_9.xml', '0000013_2_3.xml', '0000031_1.xml', '0000003_4.xml', '0000013_3.xml', '0000024_5.xml', '0000027_3.xml', '0000009_1.xml', '0000028_3.xml', '0000016_1.xml', '0000006_1.xml', '0000005_1.xml', '0000024_2.xml', '0000007_4.xml', '0000010_1.xml', '

In [15]:
import xml.etree.ElementTree as ET
import pandas as pd
import os

def parse_medquad_xml(file_path):
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
    except ET.ParseError as e:
        # print(f"Error parsing XML file {file_path}: {e}")
        return None

    doc_id = root.get('id')
    source = root.get('source')
    url = root.get('url')
    focus = root.find('Focus').text if root.find('Focus') is not None else None

    questions = []
    answers = []

    # Attempt to find explicit Q&A pairs first
    found_explicit_qa = False
    # Corrected XPath to directly find QAPair elements
    for qa_pair in root.findall('.//QAPair'):
        question = qa_pair.find('Question').text if qa_pair.find('Question') is not None else None
        answer = qa_pair.find('Answer').text if qa_pair.find('Answer') is not None else None

        if question and answer:
            questions.append(question.strip())
            answers.append(answer.strip())
            found_explicit_qa = True

    # If no explicit Q&A pairs, try to use Focus as question and DocumentText as answer
    if not found_explicit_qa:
        document_text_elem = root.find('DocumentText')
        if focus and document_text_elem is not None and document_text_elem.text:
            questions.append(focus.strip())
            answers.append(document_text_elem.text.strip())
        # If still no Q&A, try to use Focus as question and Summary content as answer
        else:
            summary_elem = root.find('Summary')
            if focus and summary_elem is not None:
                # Get text from Summary and its children, handling potential HTML/XML tags within summary
                # Create a temporary element to hold the summary content for text extraction
                temp_root = ET.Element('temp')
                temp_root.append(summary_elem)
                summary_text = ''.join(temp_root.itertext()).strip()

                if summary_text:
                    questions.append(focus.strip())
                    answers.append(summary_text)


    return {
        'doc_id': doc_id,
        'source': source,
        'url': url,
        'focus': focus,
        'questions': questions,
        'answers': answers
    }

# --- New code to find a suitable example file ---
gard_qa_path = os.path.join(medquad_path, '2_GARD_QA')

if os.path.exists(gard_qa_path):
    print(f"Searching for Q&A files in {gard_qa_path}...")
    found_qa_file = None
    for filename in os.listdir(gard_qa_path):
        if filename.endswith('.xml'):
            file_path = os.path.join(gard_qa_path, filename)
            parsed_data = parse_medquad_xml(file_path)
            if parsed_data and parsed_data['questions']:
                print(f"  Found {len(parsed_data['questions'])} Q&A pairs in {filename}")
                found_qa_file = file_path
                break # Found a file with Q&A, let's use it
            # else:
            #     print(f"  No Q&A pairs in {filename}")

    if found_qa_file:
        print(f"\nUsing {os.path.basename(found_qa_file)} as an example for detailed parsing:")
        parsed_data = parse_medquad_xml(found_qa_file)
        if parsed_data:
            print("Parsed Data from example file:")
            print(f"  Document ID: {parsed_data['doc_id']}")
            print(f"  Source: {parsed_data['source']}")
            print(f"  Focus: {parsed_data['focus']}")
            print(f"  Number of Q&A pairs found: {len(parsed_data['questions'])}")
            if parsed_data['questions']:
                print("  First Question:", parsed_data['questions'][0])
                print("  First Answer:", parsed_data['answers'][0][:200], "...") # Print first 200 chars
            else:
                print("  No explicit Q&A pairs found in this file.")
    else:
        print("No XML file with explicit Q&A pairs found in the '2_GARD_QA' directory.")
else:
    print(f"The directory {gard_qa_path} does not exist.")

Searching for Q&A files in ./MedQuAD/2_GARD_QA...
  Found 1 Q&A pairs in 0000618.xml

Using 0000618.xml as an example for detailed parsing:
Parsed Data from example file:
  Document ID: 0000618
  Source: GARD
  Focus: Bantu siderosis
  Number of Q&A pairs found: 1
  First Question: What are the symptoms of Bantu siderosis ?
  First Answer: What are the signs and symptoms of Bantu siderosis? The Human Phenotype Ontology provides the following list of signs and symptoms for Bantu siderosis. If the information is available, the table below ...


In [16]:
import os
import pandas as pd
import xml.etree.ElementTree as ET

# Re-using the parse_medquad_xml function defined previously
# (Ensuring it's available in the current context or redefining if needed)
# If you rerun this notebook from scratch, make sure the function definition cell also runs.

all_qa_data = []

# List of directories to process (excluding .git, LICENSE.txt, readme.txt, and the zip file)
subdirs_to_process = [d for d in os.listdir(medquad_path)
                      if os.path.isdir(os.path.join(medquad_path, d)) and
                         not d.startswith('.') and
                         'QA' in d.upper()] # Heuristic to pick QA directories

print(f"Processing {len(subdirs_to_process)} Q&A source directories...")

for subdir in subdirs_to_process:
    subdir_path = os.path.join(medquad_path, subdir)
    # print(f"  Entering directory: {subdir_path}")

    for filename in os.listdir(subdir_path):
        if filename.endswith('.xml'):
            file_path = os.path.join(subdir_path, filename)
            parsed_data = parse_medquad_xml(file_path)

            if parsed_data and parsed_data['questions']:
                for q, a in zip(parsed_data['questions'], parsed_data['answers']):
                    all_qa_data.append({
                        'document_id': parsed_data['doc_id'],
                        'source': parsed_data['source'],
                        'focus': parsed_data['focus'],
                        'question': q,
                        'answer': a
                    })

qa_df = pd.DataFrame(all_qa_data)

print(f"\nTotal Q&A pairs extracted: {len(qa_df)}")
print("\nFirst 5 rows of the Q&A DataFrame:")
print(qa_df.head())

if not qa_df.empty:
    print("\nQ&A pair distribution by source:")
    print(qa_df['source'].value_counts())
else:
    print("\nNo Q&A data extracted to show distribution by source.")

Processing 12 Q&A source directories...

Total Q&A pairs extracted: 16407

First 5 rows of the Q&A DataFrame:
  document_id source            focus  \
0     0000067  NHLBI  Hemochromatosis   
1     0000067  NHLBI  Hemochromatosis   
2     0000067  NHLBI  Hemochromatosis   
3     0000067  NHLBI  Hemochromatosis   
4     0000067  NHLBI  Hemochromatosis   

                                     question  \
0             What is (are) Hemochromatosis ?   
1               What causes Hemochromatosis ?   
2       Who is at risk for Hemochromatosis? ?   
3  What are the symptoms of Hemochromatosis ?   
4           How to diagnose Hemochromatosis ?   

                                              answer  
0  Hemochromatosis (HE-mo-kro-ma-TO-sis) is a dis...  
1  The two types of hemochromatosis are primary a...  
2  Hemochromatosis is one of the most common gene...  
3  Hemochromatosis can affect many parts of the b...  
4  Your doctor will diagnose hemochromatosis base...  

Q&A pair distribu

In [11]:
import os
import xml.etree.ElementTree as ET

ghr_qa_path = os.path.join(medquad_path, '3_GHR_QA')

if os.path.exists(ghr_qa_path):
    print(f"Contents of {ghr_qa_path} directory:")
    ghr_files = [f for f in os.listdir(ghr_qa_path) if f.endswith('.xml')]
    print(ghr_files[:5]) # Print first 5 files

    if ghr_files:
        example_file_path = os.path.join(ghr_qa_path, ghr_files[0]) # Take the first XML file
        print(f"\nReading full content of: {os.path.basename(example_file_path)}\n")
        try:
            with open(example_file_path, 'r', encoding='utf-8') as f:
                full_content = f.read()
                print(full_content)
            print("\n--- End of file content ---\n")
        except Exception as e:
            print(f"Error reading file {example_file_path}: {e}")
    else:
        print(f"No XML files found in {ghr_qa_path}.")
else:
    print(f"The directory {ghr_qa_path} does not exist.")

Contents of ./MedQuAD/3_GHR_QA directory:
['0000067.xml', '0000192.xml', '0000618.xml', '0001091.xml', '0000269.xml']

Reading full content of: 0000067.xml

<?xml version="1.0" encoding="UTF-8"?>
<Document id="0000067" source="GHR" url="https://ghr.nlm.nih.gov/condition/argininosuccinic-aciduria">
<Focus>argininosuccinic aciduria</Focus>
<FocusAnnotations>
	<UMLS>
		<CUIs>
			<CUI>C0268547</CUI>
		</CUIs>
		<SemanticTypes>
			<SemanticType>T047</SemanticType>
		</SemanticTypes>
		<SemanticGroup>Disorders</SemanticGroup>
	</UMLS>
	<Synonyms>
		<Synonym>Argininosuccinate lyase deficiency</Synonym>
		<Synonym>argininosuccinic acidemia</Synonym>
		<Synonym>Argininosuccinicaciduria</Synonym>
		<Synonym>argininosuccinyl-CoA lyase deficiency</Synonym>
		<Synonym>arginosuccinase deficiency</Synonym>
		<Synonym>ASA</Synonym>
		<Synonym>ASAuria</Synonym>
		<Synonym>ASL deficiency</Synonym>
	</Synonyms>
</FocusAnnotations>
<QAPairs>
	<QAPair pid="1">
			<Question qid="0000067-1" qtype="informatio

In [14]:
import xml.etree.ElementTree as ET
import os

def debug_parse_medquad_xml(file_path):
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
    except ET.ParseError as e:
        print(f"Debug: Error parsing XML file {file_path}: {e}")
        return None

    doc_id = root.get('id')
    source = root.get('source')
    url = root.get('url')
    focus = root.find('Focus').text if root.find('Focus') is not None else None

    questions = []
    answers = []

    print(f"Debug: Processing Document ID: {doc_id}, Source: {source}, Focus: {focus}")

    # Attempt to find explicit Q&A pairs first
    found_explicit_qa = False
    # Modified XPath to directly find QAPair elements
    qa_pairs_found_in_xml = root.findall('.//QAPair')
    print(f"Debug: Found {len(qa_pairs_found_in_xml)} QAPair elements using .//QAPair XPath.")

    for i, qa_pair in enumerate(qa_pairs_found_in_xml):
        question_elem = qa_pair.find('Question')
        answer_elem = qa_pair.find('Answer')

        question = question_elem.text if question_elem is not None else None
        answer = answer_elem.text if answer_elem is not None else None

        print(f"Debug:   QAPair {i+1}: Question element: {question_elem is not None}, Answer element: {answer_elem is not None}")
        print(f"Debug:     Question text: {question[:50] + '...' if question else 'None'}")
        print(f"Debug:     Answer text: {answer[:50] + '...' if answer else 'None'}")

        if question and answer:
            questions.append(question.strip())
            answers.append(answer.strip())
            found_explicit_qa = True

    print(f"Debug: After explicit QA search, questions list has {len(questions)} items. found_explicit_qa: {found_explicit_qa}")

    # If no explicit Q&A pairs, try to use Focus as question and DocumentText as answer
    if not found_explicit_qa:
        document_text_elem = root.find('DocumentText')
        print(f"Debug: No explicit QA found. Checking DocumentText. DocumentText exists: {document_text_elem is not None}")
        if focus and document_text_elem is not None and document_text_elem.text:
            questions.append(focus.strip())
            answers.append(document_text_elem.text.strip())
            print("Debug: Added Q&A from Focus and DocumentText.")
        # If still no Q&A, try to use Focus as question and Summary content as answer
        else:
            summary_elem = root.find('Summary')
            print(f"Debug: No DocumentText Q&A. Checking Summary. Summary exists: {summary_elem is not None}")
            if focus and summary_elem is not None:
                temp_root = ET.Element('temp')
                temp_root.append(summary_elem)
                summary_text = ''.join(temp_root.itertext()).strip()

                if summary_text:
                    questions.append(focus.strip())
                    answers.append(summary_text)
                    print("Debug: Added Q&A from Focus and Summary.")

    print(f"Debug: Final questions list size: {len(questions)}")
    return {
        'doc_id': doc_id,
        'source': source,
        'url': url,
        'focus': focus,
        'questions': questions,
        'answers': answers
    }

# Test with the known good example file
example_ghr_file = './MedQuAD/3_GHR_QA/0000067.xml'
print(f"\n--- Starting Debugging for {example_ghr_file} ---")
debug_parsed_data = debug_parse_medquad_xml(example_ghr_file)

if debug_parsed_data:
    print(f"\nDebug: Final Parsed Data (questions count): {len(debug_parsed_data['questions'])}")
    if debug_parsed_data['questions']:
        print("Debug: First Question:", debug_parsed_data['questions'][0][:100], "...")
        print("Debug: First Answer:", debug_parsed_data['answers'][0][:100], "...")
    else:
        print("Debug: No questions extracted by debug function.")
else:
    print("Debug: Parsing returned None.")


--- Starting Debugging for ./MedQuAD/3_GHR_QA/0000067.xml ---
Debug: Processing Document ID: 0000067, Source: GHR, Focus: argininosuccinic aciduria
Debug: Found 5 QAPair elements using .//QAPair XPath.
Debug:   QAPair 1: Question element: True, Answer element: True
Debug:     Question text: What is (are) argininosuccinic aciduria ?...
Debug:     Answer text: Argininosuccinic aciduria is an inherited disorder...
Debug:   QAPair 2: Question element: True, Answer element: True
Debug:     Question text: How many people are affected by argininosuccinic a...
Debug:     Answer text: Argininosuccinic aciduria occurs in approximately ...
Debug:   QAPair 3: Question element: True, Answer element: True
Debug:     Question text: What are the genetic changes related to argininosu...
Debug:     Answer text: Mutations in the ASL gene cause argininosuccinic a...
Debug:   QAPair 4: Question element: True, Answer element: True
Debug:     Question text: Is argininosuccinic aciduria inherited ?...
Debug: